# AE1 · Cuaderno base: búsqueda no informada

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Actividad Evaluativa 1, parte B**

Este cuaderno es una **estructura sugerida**, no una solución. Contiene:

1. Una interfaz común `Problema` para que los cuatro algoritmos funcionen con cualquier problema.
2. La clase `Nodo` de la práctica de la semana 2.
3. Un registro de métricas: nodos expandidos, tamaño máximo de la frontera y tiempo.
4. **BFS ya implementada** como ejemplo de cómo registrar las métricas.
5. Los esqueletos de DFS, profundidad iterativa y coste uniforme, que **debes completar tú**.
6. Un ejemplo con el puzle lineal de la semana 2, que debes **reemplazar por tu problema**.
7. La tabla y la gráfica de resultados.

Busca las marcas `# TODO` para saber qué te toca escribir. Cambia el título de este cuaderno y guárdalo en tu repositorio como `ae1/ae1-busqueda.ipynb`.

## 1. La interfaz `Problema`

Cada problema se describe con los cuatro componentes de Russell y Norvig (2004, cap. 3): estado inicial, acciones (con su modelo de transición), test objetivo y costo. Los algoritmos solo usan estos métodos, así que no tienes que modificarlos cuando cambies de problema.

Los estados deben ser **inmutables** (tuplas, cadenas, números o `frozenset`) para poder guardarlos en un `set`.

In [ ]:
import time
import heapq
from collections import deque


class Problema:
    # Interfaz común. Tu problema debe heredar de esta clase.

    def __init__(self, inicial):
        self.inicial = inicial

    def acciones(self, estado):
        # Lista de acciones aplicables en el estado.
        raise NotImplementedError

    def resultado(self, estado, accion):
        # Estado que se obtiene al aplicar la acción (modelo de transición).
        raise NotImplementedError

    def es_objetivo(self, estado):
        raise NotImplementedError

    def costo(self, estado, accion, siguiente):
        # Costo individual c(x, a, y). Por omisión, 1 por acción.
        return 1

    def sucesores(self, estado):
        # Pares (acción, estado siguiente), como la función sucesor del capítulo 2.
        return [(a, self.resultado(estado, a)) for a in self.acciones(estado)]

## 2. La estructura `Nodo`

Igual a la de la práctica de la semana 2: estado, padre, acción, costo del camino $g(n)$ y profundidad.

In [ ]:
class Nodo:
    def __init__(self, estado, padre=None, accion=None, costo=0):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.costo = costo
        self.profundidad = 0 if padre is None else padre.profundidad + 1

    def camino(self):
        nodo, estados = self, []
        while nodo is not None:
            estados.append(nodo.estado)
            nodo = nodo.padre
        return list(reversed(estados))

    def acciones(self):
        nodo, acciones = self, []
        while nodo.padre is not None:
            acciones.append(nodo.accion)
            nodo = nodo.padre
        return list(reversed(acciones))

    def __lt__(self, otro):                  # desempate en heapq
        return self.costo < otro.costo

## 3. Registro de métricas

Cada algoritmo recibe un objeto `Metricas` y lo actualiza:

- `m.expandido()` cada vez que expande un nodo.
- `m.frontera(n)` cada vez que cambia el tamaño de la frontera (guarda el máximo).

La función `ejecutar` mide el tiempo y corta la búsqueda si se supera un límite de nodos expandidos, para que DFS no se quede corriendo indefinidamente en una instancia difícil.

In [ ]:
class LimiteSuperado(Exception):
    pass


class Metricas:
    def __init__(self, limite_expandidos):
        self.expandidos = 0
        self.max_frontera = 0
        self.limite = limite_expandidos

    def expandido(self):
        self.expandidos += 1
        if self.expandidos > self.limite:
            raise LimiteSuperado()

    def frontera(self, tamano):
        self.max_frontera = max(self.max_frontera, tamano)


def ejecutar(algoritmo, problema, limite_expandidos=1_000_000):
    m = Metricas(limite_expandidos)
    t0 = time.perf_counter()
    try:
        nodo = algoritmo(problema, m)
        estado = "solución" if nodo is not None else "sin solución"
    except LimiteSuperado:
        nodo, estado = None, "límite superado"
    except NotImplementedError:
        nodo, estado = None, "no implementado"
    t = time.perf_counter() - t0
    return {
        "algoritmo": algoritmo.__name__,
        "resultado": estado,
        "longitud": nodo.profundidad if nodo else None,
        "costo": nodo.costo if nodo else None,
        "expandidos": m.expandidos,
        "max_frontera": m.max_frontera,
        "tiempo_s": round(t, 4),
        "nodo": nodo,
    }

## 4. Los algoritmos

### 4.1 Búsqueda en amplitud (BFS) · ejemplo completo

Observa dónde se llaman `m.expandido()` y `m.frontera(...)`. Esta versión aplica el test objetivo al **sacar** el nodo de la frontera, como en la sección 2.5 del capítulo.

In [ ]:
def bfs(problema, m):
    frontera = deque([Nodo(problema.inicial)])
    en_frontera = {problema.inicial}
    visitados = set()
    m.frontera(len(frontera))
    while frontera:
        nodo = frontera.popleft()
        en_frontera.discard(nodo.estado)
        if problema.es_objetivo(nodo.estado):
            return nodo
        visitados.add(nodo.estado)
        m.expandido()
        for accion, hijo in problema.sucesores(nodo.estado):
            if hijo not in visitados and hijo not in en_frontera:
                c = nodo.costo + problema.costo(nodo.estado, accion, hijo)
                frontera.append(Nodo(hijo, nodo, accion, c))
                en_frontera.add(hijo)
        m.frontera(len(frontera))
    return None

### 4.2 Búsqueda en profundidad (DFS)

Pista: el cambio principal respecto a BFS está en la estructura de la frontera (sección 2.6).

In [ ]:
def dfs(problema, m):
    # TODO: implementa la búsqueda en profundidad con lista de visitados.
    raise NotImplementedError

### 4.3 Búsqueda con profundidad iterativa

Pista: escribe primero una búsqueda en profundidad **limitada** que devuelva el nodo objetivo, `"corte"` o `None` (sección 2.6.3), y luego repítela con límites 0, 1, 2, … (sección 2.6.4). Usa el mismo objeto `m` en todas las iteraciones para que se sumen los nodos expandidos. Como esta búsqueda no tiene una frontera explícita, registra con `m.frontera(...)` la profundidad del nodo actual: es la medida de su memoria, *O*(*bd*).

In [ ]:
def profundidad_iterativa(problema, m, limite_max=100):
    # TODO: implementa la búsqueda con profundidad iterativa.
    raise NotImplementedError

### 4.4 Búsqueda de coste uniforme (UCS)

Pista: la frontera es una cola con prioridad ordenada por $g(n)$ (`heapq`), y el test objetivo se aplica al **sacar** el nodo de la frontera (sección 2.7).

In [ ]:
def coste_uniforme(problema, m):
    # TODO: implementa la búsqueda de coste uniforme.
    raise NotImplementedError

## 5. Tu problema

Abajo está el **puzle lineal** de la semana 2 solo como ejemplo de cómo heredar de `Problema`. **Reemplázalo por tu problema** y documenta en una celda de texto la formulación que escribiste en el informe.

In [ ]:
class PuzleLineal(Problema):
    # Ejemplo: reemplázalo por tu problema.
    OBJETIVO = (1, 2, 3, 4)

    def acciones(self, estado):
        return ["I", "C", "D"]

    def resultado(self, estado, accion):
        i = {"I": 0, "C": 1, "D": 2}[accion]
        e = list(estado)
        e[i], e[i + 1] = e[i + 1], e[i]
        return tuple(e)

    def es_objetivo(self, estado):
        return estado == self.OBJETIVO


r = ejecutar(bfs, PuzleLineal((4, 2, 3, 1)))
print(r["resultado"], "| acciones:", r["nodo"].acciones(), "| expandidos:", r["expandidos"])

## 6. Experimentos

Define al menos tres instancias de dificultad creciente y ejecuta los cuatro algoritmos sobre cada una.

In [ ]:
# TODO: reemplaza por instancias de tu problema (fácil, media y difícil).
instancias = {
    "fácil": PuzleLineal((1, 3, 2, 4)),
    "media": PuzleLineal((3, 1, 4, 2)),
    "difícil": PuzleLineal((4, 3, 2, 1)),
}
algoritmos = [bfs, dfs, profundidad_iterativa, coste_uniforme]

resultados = []
for nombre, problema in instancias.items():
    for alg in algoritmos:
        r = ejecutar(alg, problema, limite_expandidos=200_000)
        r["instancia"] = nombre
        resultados.append(r)

In [ ]:
import pandas as pd

tabla = pd.DataFrame(resultados).drop(columns="nodo")
tabla = tabla[["instancia", "algoritmo", "resultado", "longitud", "costo",
               "expandidos", "max_frontera", "tiempo_s"]]
tabla

In [ ]:
import matplotlib.pyplot as plt

datos = tabla[tabla["resultado"] == "solución"]
pivote = datos.pivot(index="instancia", columns="algoritmo", values="expandidos")
pivote = pivote.reindex([i for i in instancias if i in pivote.index])
ax = pivote.plot(kind="bar", logy=True, figsize=(8, 4), rot=0)
ax.set_ylabel("Nodos expandidos (escala logarítmica)")
ax.set_title("Nodos expandidos por algoritmo e instancia")
plt.tight_layout()
plt.show()

## 7. Análisis

Escribe aquí, o en el informe, las respuestas a las preguntas de la sección B.4 del enunciado. Apóyate en la tabla y en la gráfica.

---

**Referencias**

García Serrano, A. (2016). *Inteligencia artificial: Fundamentos, práctica y aplicaciones* (2.ª ed.). RC Libros.

Russell, S. J., y Norvig, P. (2004). *Inteligencia artificial: Un enfoque moderno* (2.ª ed.; J. M. Corchado Rodríguez et al., Trads.). Pearson Educación.